# BNNR — ResNet50 / CIFAR-100 augmentation benchmark (Colab)

Reproducible comparison of four augmentation strategies on a real architecture:

| Condition | Augmentation |
|-----------|--------------|
| `no_bnnr` | Crop + flip only |
| `randaugment` | + torchvision RandAugment |
| `trivialaugment` | + torchvision TrivialAugmentWide |
| `bnnr_branch_search` | Full BNNR loop (ICD + AICD + ChurchNoise) |

**5 seeds × 4 conditions = 20 runs.** ETA on T4: ~2h. ETA on A100: ~30 min.

**Resume-safe.** Results stream to Google Drive after every (condition, seed) — if the Colab session dies, just re-run all cells: completed runs are skipped automatically.

**What lands on your Drive:**
- `bnnr_benchmarks/results_resnet50.json` — aggregated metrics
- `bnnr_benchmarks/runs_resnet50/<run_dir>/xai/*.png` — OptiCAM overlays
- `bnnr_benchmarks/runs_resnet50.zip` — full archive (created in last cell)


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/bnnr_benchmarks'
!mkdir -p {DRIVE_BASE}
print(f'Drive output directory: {DRIVE_BASE}')


## 2. Clone repo and install BNNR


In [ ]:
%cd /content
![ -d bnnr ] || git clone --depth=1 https://github.com/bnnr-team/bnnr.git
%cd /content/bnnr
!git pull --rebase
!pip install -e . -q
import os; print('BNNR installed from:', os.getcwd())


## 3. GPU check + ETA

Confirm Colab gave you a GPU. T4 is fine but slow (~2h). If you got a CPU runtime, go to **Runtime → Change runtime type → T4 GPU** and re-run from the top.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}')
    eta = {'T4': '~2h', 'A100': '~30 min', 'V100': '~1h', 'L4': '~1h'}
    for k, v in eta.items():
        if k in name:
            print(f'ETA: {v} for full 5-seed run')
            break
else:
    print('WARNING: no GPU. Full run will take 10+ hours on CPU. Switch runtime.')


## 4. Dry-run preview

Prints the planned matrix without running anything. Use this to verify the config before you commit to a 2h run.


In [ ]:
!PYTHONPATH=src python benchmarks/run_resnet50.py \
  --seeds 42,43,44,45,46 \
  --device cuda \
  --drive-base-dir {DRIVE_BASE} \
  --dry-run


## 5. Smoke test (2 minutes)

1 epoch, 256/128 subset, img-size 64, no pretrained download. Verifies the full pipeline (data load → train → XAI export → JSON save) on real GPU before committing to the long run.

Smoke results write to `{DRIVE_BASE}/results_resnet50.json`. **Delete that file after the smoke if you want to start fresh** — see cell 6 cleanup.


In [ ]:
!PYTHONPATH=src python benchmarks/run_resnet50.py \
  --device cuda \
  --drive-base-dir {DRIVE_BASE} \
  --smoke


## 6. (Optional) Clean smoke results before full run

If you ran the smoke in cell 5, the smoke entries are mixed into `results_resnet50.json`. **Run this cell to wipe them** so the full run starts from a clean slate. Skip this cell if you ran the smoke locally before and want to keep partial real results.


In [ ]:
import json, os
results_path = f'{DRIVE_BASE}/results_resnet50.json'
if os.path.exists(results_path):
    with open(results_path) as f: data = json.load(f)
    before = len(data.get('runs', []))
    # Drop smoke entries (img_size=64, m_epochs=1) and error entries; keep everything else
    def _is_smoke(r):
        return r.get('img_size') == 64 or r.get('m_epochs', 0) == 1
    data['runs'] = [r for r in data.get('runs', []) if not _is_smoke(r) and 'val_metric' in r]
    after = len(data['runs'])
    with open(results_path, 'w') as f: json.dump(data, f, indent=2)
    print(f'Filtered {before} -> {after} runs (dropped smoke + error entries)')
else:
    print('No results file yet — nothing to clean.')


## 7. Full benchmark — 5 seeds × 4 conditions

**This is the long run.** Resume-safe: if Colab kills the session, just re-run this cell. Already-completed (condition, seed) pairs will be skipped.

Keep the browser tab active. Colab idles out after ~90 min of no interaction. To stay alive: occasionally scroll, or use a Colab keep-alive browser extension.


In [ ]:
!PYTHONPATH=src python benchmarks/run_resnet50.py \
  --seeds 42,43,44,45,46 \
  --device cuda \
  --drive-base-dir {DRIVE_BASE}


## 8. Summarize → markdown table


In [ ]:
!PYTHONPATH=src python benchmarks/summarize.py \
  --results {DRIVE_BASE}/results_resnet50.json \
  --markdown


## 8b. Validate results completeness

Check that all 20 runs completed without errors before proceeding.


In [ ]:
import json, os
results_path = f'{DRIVE_BASE}/results_resnet50.json'
with open(results_path) as f: data = json.load(f)
runs = data.get('runs', [])
errors = [r for r in runs if 'error' in r]
valid = [r for r in runs if 'val_metric' in r and 'error' not in r]
conditions = sorted(set(r['condition'] for r in valid))
seeds = sorted(set(r['seed'] for r in valid))
print(f'Total records : {len(runs)}')
print(f'Valid runs    : {len(valid)} / 20 expected')
print(f'Errors        : {len(errors)}')
print(f'Conditions    : {conditions}')
print(f'Seeds         : {seeds}')
if errors:
    print('\nFailed runs:')
    for e in errors:
        print(f"  {e['condition']} seed={e['seed']}: {e.get('error','?')}")
missing = []
for cond in ['no_bnnr', 'randaugment', 'trivialaugment', 'bnnr_branch_search']:
    for seed in [42, 43, 44, 45, 46]:
        if not any(r['condition']==cond and r['seed']==seed for r in valid):
            missing.append((cond, seed))
if missing:
    print(f'\nMissing runs ({len(missing)}):')
    for cond, seed in missing:
        print(f'  {cond} seed={seed}')
    print('\nRe-run cell 7 to fill in missing runs automatically.')
else:
    print('\n✓ All 20 runs present and valid.')


## 9. Backup `runs_resnet50/` directory as ZIP on Drive

The `runs_resnet50/` directory contains all XAI overlay PNGs and per-run logs. The full run already wrote those to Drive (because of `--drive-base-dir`), but a single ZIP is faster to download and easier to share.


In [ ]:
import shutil
zip_path = shutil.make_archive(
    f'{DRIVE_BASE}/runs_resnet50_archive',
    'zip',
    f'{DRIVE_BASE}/runs_resnet50',
)
print(f'Archive created: {zip_path}')
!ls -lh {DRIVE_BASE}/


## Done

You now have on your Drive:
- `results_resnet50.json` — aggregated metrics for all 20 runs
- `runs_resnet50/<run_dir>/train.log` — epoch-by-epoch training log per run
- `runs_resnet50/<run_dir>/xai/*.png` — OptiCAM overlays per run
- `runs_resnet50/<run_dir>/run_summary.json` — per-run best metrics and XAI stats
- `runs_resnet50_archive.zip` — full backup

**Next steps:**
1. Run cell 8b to confirm all 20 runs are present and error-free
2. Copy the markdown table from cell 8 into `benchmarks/README.md` § Results
3. Pick the best XAI overlay comparison from `runs_resnet50/*/xai/` for the README hero image
